# 60 Visualize MediaPipe detections on S8

Render Subject 12 (S8, bare-cap) from the same 21-view sweep used in notebooks 56 and 59, run MediaPipe Face Detector on every view, and draw the detected bounding box (with confidence score) on top of each rendered view. This makes it possible to see exactly what BlazeFace is locking onto when it fires on the bare-cap cohort.

Three variants are rendered for visual comparison:

- **Original**: head-isolated CTF mesh straight out of `run_pipeline` (no anonymization).
- **Deletion**: same mesh after `delete_masked_vertices` (the shipped operator).
- **Noise**: same mesh after the rejected noise-perturbation strawman (`noise_perturb_with_taper`).

All three share head-isolation and CTF alignment from the same in-memory pipeline run, so the only factor that varies across variants is the operator applied. Detection bounding boxes are drawn in green; views with no detection are left clean.

Output: `thesis_results_out/s8_mediapipe_boxes/{variant}_grid.png` (3 contact sheets, one per variant).

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))
from _thesis_helpers import (
    s_id, load_raw, load_landmarks, run_pipeline,
    noise_perturb_with_taper,
)

import numpy as np
import pandas as pd
import pyvista as pv
import cv2
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import mediapipe as mp
from mediapipe.tasks import python as mp_py
from mediapipe.tasks.python import vision as mp_vision

import cedalion.dataclasses as cdc
from cedalion.vtktutils import trimesh_to_vtk_polydata

pv.OFF_SCREEN = True

SUBJECT = 12  # S8
OUT_DIR = pathlib.Path('thesis_results_out') / 's8_mediapipe_boxes'
OUT_DIR.mkdir(parents=True, exist_ok=True)

YAWS = list(range(-90, 91, 30))   # 7 yaws
PITCHES = [-20, 0, 20]            # 3 pitches
WINDOW = (640, 640)
GREY = (200, 200, 200)
CAM_DISTANCE_MM = 700.0
ZOOM = 1.3

MODEL_PATH = str((pathlib.Path('models') / 'blaze_face_short_range.tflite').resolve())
_fd = mp_vision.FaceDetector.create_from_options(
    mp_vision.FaceDetectorOptions(
        base_options=mp_py.BaseOptions(model_asset_path=MODEL_PATH),
        min_detection_confidence=0.5,
    )
)

I0000 00:00:1777404200.235939   58491 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1777404200.244164   58516 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.0.4-arch1.1), renderer: AMD Radeon Graphics (radeonsi, renoir, ACO, DRM 3.64, 6.19.11-arch1-1)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1777404200.249578   58495 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


## 1. Render sweep

Reuses the same render geometry as notebook 59: camera placed on +X (anterior) at 700 mm, looking at the head centroid, +Z up. yaw=0 / pitch=0 is true frontal.

In [2]:
def render_views(surface, out_dir, tag):
    out_dir.mkdir(parents=True, exist_ok=True)
    poly = pv.wrap(trimesh_to_vtk_polydata(surface.mesh))
    focal = np.asarray(surface.mesh.vertices).mean(axis=0)
    files = []
    for yaw_deg in YAWS:
        for pitch_deg in PITCHES:
            yaw = np.radians(yaw_deg)
            pitch = np.radians(pitch_deg)
            # Camera position in CTF frame: +X anterior, +Y left, +Z up.
            x = CAM_DISTANCE_MM * np.cos(pitch) * np.cos(yaw)
            y = CAM_DISTANCE_MM * np.cos(pitch) * np.sin(yaw)
            z = CAM_DISTANCE_MM * np.sin(pitch)
            cam = focal + np.array([x, y, z])

            pl = pv.Plotter(off_screen=True, window_size=WINDOW)
            pl.add_mesh(poly, color=[c/255 for c in GREY], smooth_shading=True)
            pl.set_background('white')
            pl.enable_anti_aliasing('ssaa')
            pl.camera.position = tuple(cam)
            pl.camera.focal_point = tuple(focal)
            pl.camera.up = (0, 0, 1)
            pl.camera.zoom(ZOOM)
            fn = out_dir / f'{tag}_yaw{yaw_deg:+04d}_pitch{pitch_deg:+03d}.png'
            pl.screenshot(str(fn))
            pl.close()
            files.append((yaw_deg, pitch_deg, fn))
    return files

## 2. Detect + draw boxes

Returns the annotated image and a list of (xmin, ymin, w, h, score) per detection.

In [3]:
def detect_and_annotate(image_path):
    img = cv2.imread(str(image_path))
    if img is None:
        return None, []
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = _fd.detect(mp_img)

    boxes = []
    annotated = img.copy()
    if result.detections:
        for det in result.detections:
            bbox = det.bounding_box
            x, y, w, h = bbox.origin_x, bbox.origin_y, bbox.width, bbox.height
            score = float(det.categories[0].score)
            boxes.append((x, y, w, h, score))
            cv2.rectangle(annotated, (x, y), (x + w, y + h), (0, 255, 0), 3)
            label = f'{score:.2f}'
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
            cv2.rectangle(annotated, (x, y - th - 8), (x + tw + 6, y), (0, 255, 0), -1)
            cv2.putText(annotated, label, (x + 3, y - 4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2)
    return annotated, boxes

## 3. Run all three variants on S8

In [4]:
raw = load_raw(SUBJECT)
lm = load_landmarks(SUBJECT)
art = run_pipeline(raw, lm, subject=SUBJECT)
lm_ctf_xyz = art.landmarks_ctf.pint.dequantify().values

noise_mesh = noise_perturb_with_taper(art.surface_ctf.mesh, art.mask, lm_ctf_xyz)
surface_noise = cdc.TrimeshSurface(
    mesh=noise_mesh,
    crs=art.surface_ctf.crs,
    units=art.surface_ctf.units,
)

variants = {
    'original': art.surface_ctf,
    'delete':   art.surface_anon_ctf,
    'noise':    surface_noise,
}

all_rows = []
annotated_paths = {tag: {} for tag in variants}

for tag, surface in variants.items():
    print(f'\n=== {tag} ===', flush=True)
    sub_dir = OUT_DIR / tag
    files = render_views(surface, sub_dir, tag)
    annot_dir = sub_dir / 'annotated'
    annot_dir.mkdir(exist_ok=True)
    for yaw, pitch, fn in files:
        annotated, boxes = detect_and_annotate(fn)
        out_fn = annot_dir / fn.name
        cv2.imwrite(str(out_fn), annotated)
        annotated_paths[tag][(yaw, pitch)] = out_fn
        if boxes:
            for (x, y, w, h, score) in boxes:
                all_rows.append({
                    's_id': s_id(SUBJECT), 'subject': SUBJECT,
                    'variant': tag, 'yaw': yaw, 'pitch': pitch,
                    'bbox_x': x, 'bbox_y': y, 'bbox_w': w, 'bbox_h': h,
                    'score': score,
                })
        print(f'  yaw={yaw:+4d} pitch={pitch:+3d}  hits={len(boxes)}'
              f'  scores={[round(b[4], 2) for b in boxes]}')

df = pd.DataFrame(all_rows)
csv_out = OUT_DIR / 's8_detections.csv'
df.to_csv(csv_out, index=False)
print(f'\nWrote {csv_out} with {len(df)} detections across the three variants.')


=== original ===


  yaw= -90 pitch=-20  hits=1  scores=[0.59]
  yaw= -90 pitch= +0  hits=0  scores=[]
  yaw= -90 pitch=+20  hits=1  scores=[0.65]
  yaw= -60 pitch=-20  hits=2  scores=[0.83, 0.57]
  yaw= -60 pitch= +0  hits=1  scores=[0.83]
  yaw= -60 pitch=+20  hits=1  scores=[0.69]


  yaw= -30 pitch=-20  hits=1  scores=[0.93]
  yaw= -30 pitch= +0  hits=1  scores=[0.91]
  yaw= -30 pitch=+20  hits=1  scores=[0.65]
  yaw=  +0 pitch=-20  hits=1  scores=[0.92]
  yaw=  +0 pitch= +0  hits=1  scores=[0.92]
  yaw=  +0 pitch=+20  hits=1  scores=[0.87]
  yaw= +30 pitch=-20  hits=1  scores=[0.95]


  yaw= +30 pitch= +0  hits=1  scores=[0.91]
  yaw= +30 pitch=+20  hits=2  scores=[0.65, 0.63]
  yaw= +60 pitch=-20  hits=1  scores=[0.83]
  yaw= +60 pitch= +0  hits=1  scores=[0.65]
  yaw= +60 pitch=+20  hits=1  scores=[0.56]
  yaw= +90 pitch=-20  hits=0  scores=[]
  yaw= +90 pitch= +0  hits=0  scores=[]


  yaw= +90 pitch=+20  hits=1  scores=[0.56]

=== delete ===


...

  yaw= +90 pitch=+20  hits=1  scores=[0.58]

=== noise ===


  yaw= -90 pitch=-20  hits=0  scores=[]
  yaw= -90 pitch= +0  hits=0  scores=[]
  yaw= -90 pitch=+20  hits=1  scores=[0.66]
  yaw= -60 pitch=-20  hits=1  scores=[0.53]
  yaw= -60 pitch= +0  hits=1  scores=[0.72]
  yaw= -60 pitch=+20  hits=0  scores=[]


  yaw= -30 pitch=-20  hits=1  scores=[0.55]
  yaw= -30 pitch= +0  hits=0  scores=[]
  yaw= -30 pitch=+20  hits=1  scores=[0.61]
  yaw=  +0 pitch=-20  hits=1  scores=[0.76]
  yaw=  +0 pitch= +0  hits=0  scores=[]
  yaw=  +0 pitch=+20  hits=1  scores=[0.76]
  yaw= +30 pitch=-20  hits=1  scores=[0.8]


  yaw= +30 pitch= +0  hits=1  scores=[0.8]
  yaw= +30 pitch=+20  hits=1  scores=[0.72]
  yaw= +60 pitch=-20  hits=0  scores=[]
  yaw= +60 pitch= +0  hits=1  scores=[0.62]
  yaw= +60 pitch=+20  hits=1  scores=[0.64]
  yaw= +90 pitch=-20  hits=0  scores=[]


  yaw= +90 pitch= +0  hits=0  scores=[]
  yaw= +90 pitch=+20  hits=1  scores=[0.57]

Wrote thesis_results_out/s8_mediapipe_boxes/s8_detections.csv with 48 detections across the three variants.


## 4. Build a 3 x 21 contact sheet

Rows: original / delete / noise. Columns: 21 viewpoints (7 yaws x 3 pitches). Green boxes mark MediaPipe detections; white panels mean no detection.

In [5]:
TAGS = ['original', 'delete', 'noise']
ROW_TITLES = ['Original', 'Vertex deletion', 'Noise perturbation']

view_keys = [(y, p) for y in YAWS for p in PITCHES]
n_cols = len(view_keys)

fig, axes = plt.subplots(
    len(TAGS), n_cols,
    figsize=(1.6 * n_cols, 1.6 * len(TAGS)),
)
for r, tag in enumerate(TAGS):
    for c, (yaw, pitch) in enumerate(view_keys):
        ax = axes[r, c]
        path = annotated_paths[tag][(yaw, pitch)]
        img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_xticks([]); ax.set_yticks([])
        if r == 0:
            ax.set_title(f'y{yaw:+d}/p{pitch:+d}', fontsize=6)
        if c == 0:
            ax.set_ylabel(ROW_TITLES[r], fontsize=10)
fig.suptitle(f'{s_id(SUBJECT)} (Subject{SUBJECT}) - MediaPipe detections on every view')
fig.tight_layout()
out = OUT_DIR / 's8_mediapipe_grid.png'
fig.savefig(out, dpi=180, bbox_inches='tight')
plt.close(fig)
print(f'Wrote {out}')

Wrote thesis_results_out/s8_mediapipe_boxes/s8_mediapipe_grid.png


## 5. Per-variant summary

In [6]:
n_views = len(YAWS) * len(PITCHES)
for tag in TAGS:
    sub = df[df.variant == tag] if len(df) else df
    n_hit_views = sub.groupby(['yaw', 'pitch']).size().shape[0] if len(sub) else 0
    n_dets = len(sub)
    max_score = float(sub.score.max()) if len(sub) else 0.0
    print(f'{tag:10s}  views_with_hit={n_hit_views:>2d}/{n_views}  '
          f'total_detections={n_dets:>2d}  max_score={max_score:.3f}')

original    views_with_hit=18/21  total_detections=20  max_score=0.946
delete      views_with_hit=15/21  total_detections=15  max_score=0.819
noise       views_with_hit=13/21  total_detections=13  max_score=0.797


In [ ]:
## Figure for the thesis (Subject 17 / S2 only, per data-sharing rule).
##
## The cell above produced the full keypoint diagnostic on Subject 12 (S8) for
## by-eye verification - that diagnostic stays as supplementary analysis. The
## thesis figure is restricted to S2 (Subject 17) and is composed below from
## annotated S2 renders saved in
## `thesis_results_out/s2_keypoint_diagnostics/`.
##
## Both the bare-cap S8 case and the optode-cap S2 case show the same pattern:
## post-deletion residual hits land on bald-skull silhouettes with clustered,
## anatomically inconsistent keypoint placements.

import shutil

S2_DIAG = pathlib.Path('thesis_results_out/s2_keypoint_diagnostics')
THESIS_FIG_DIR = pathlib.Path(
    '/home/ma7/BA/Thesis_template-master-LaTeX_template_thesis/'
    'LaTeX_template_thesis/Figures/results'
)

panels = [
    (S2_DIAG / 'A_real_face_S2.png',
     r'Original, yaw $0^{\circ}$, pitch $-20^{\circ}$' '\n' r'score 0.82 - real face'),
    (S2_DIAG / 'B_hallucination_S2.png',
     r'Deletion, yaw $+60^{\circ}$, pitch $-20^{\circ}$' '\n' r'score 0.52 - hallucination'),
]

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, (path, title) in zip(axes, panels):
    img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title, fontsize=10)
fig.tight_layout()

out = THESIS_FIG_DIR / 'detectability_keypoints_S2.png'
fig.savefig(out, dpi=200, bbox_inches='tight')
plt.close(fig)
print(f'Wrote {out}')

## 6. Two-panel keypoint diagnostic for the thesis

A 2-panel figure for §4.4.4: a real-face hit (frontal, original) where the BlazeFace 6 keypoints land on real eyes / nose / mouth / ears, side-by-side with a hallucination (left profile, deletion variant) where the keypoints cluster onto the bald-skull silhouette in an anatomically impossible configuration (nose tip above eyes, mouth at eye level). Composed from the previously-saved `keypoint_diagnostics/A_*` and `keypoint_diagnostics/C_*` PNGs. Saves directly into the LaTeX `Figures/results/` tree (`feedback_pngs_in_latex_dir`).